# LeetCode 978 — Longest Turbulent Subarray bằng Python

Link đề: https://leetcode.com/problems/longest-turbulent-subarray/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def maxTurbulenceSize(self, arr: List[int]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên** (độ dài), duy nhất. Điểm cốt lõi: đoạn con "turbulent" là đoạn mà dấu so sánh giữa các cặp liền kề **đổi chiều liên tục** (`<`, `>`, `<`, … hoặc `>`, `<`, `>`, …); cặp **bằng nhau** phá vỡ đoạn. Một phần tử đơn lẻ luôn là đoạn turbulent (độ dài 1) nên đáp án luôn `≥ 1`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm độ dài lớn nhất của đoạn con liên tiếp trong đó các so sánh liền kề **xen kẽ** `<` / `>`.
2. Nắm công cụ Python: mảng dấu `sign(arr[i+1] - arr[i])`, quy hoạch động hai trạng thái (kết thúc bằng bước tăng / bước giảm), cửa sổ trượt, `itertools.groupby`.
3. Đi từ **thử mọi đoạn (theo định nghĩa)** → **DP hai trạng thái `up`/`down`** → **cửa sổ trượt một lượt** → **biến đổi dấu theo chẵn lẻ + `groupby`** → **chia để trị (gộp tiền tố/hậu tố)**.
4. Hiểu mấu chốt: gọi `d[i] = sign(arr[i+1] − arr[i]) ∈ {−1, 0, +1}`; đoạn `arr[l..r]` turbulent ⇔ `d[l..r−1]` đều **khác 0** và **xen kẽ dấu**; đáp án = (độ dài run xen kẽ dài nhất của `d`) + 1, hoặc `1` nếu không có.
5. Tránh lỗi kinh điển: coi cặp **bằng nhau** là hợp lệ (`≤`/`≥`), trả `0` cho mảng một phần tử hoặc toàn số bằng nhau, reset cửa sổ sai (đặt `start = i` thay vì `i − 1`), nhầm "số cặp" với "số phần tử" (lệch 1), chỉ xét một hướng bắt đầu.

## 2. Đề bài & ví dụ

Cho mảng `arr`. Trả về độ dài lớn nhất của một **đoạn con turbulent**.

Đoạn con `arr[i..j]` là turbulent nếu dấu so sánh đổi chiều giữa mọi cặp liền kề. Chính xác hơn: hoặc với mọi `i ≤ k < j`, `arr[k] > arr[k+1]` khi `k` lẻ và `arr[k] < arr[k+1]` khi `k` chẵn; hoặc với mọi `i ≤ k < j`, `arr[k] > arr[k+1]` khi `k` chẵn và `arr[k] < arr[k+1]` khi `k` lẻ.

**Ví dụ 1:** `arr = [9,4,2,10,7,8,8,1,9]` → `5` (`arr[1] > arr[2] < arr[3] > arr[4] < arr[5]`).

**Ví dụ 2:** `arr = [4,8,12,16]` → `2`.

**Ví dụ 3:** `arr = [100]` → `1`.

**Ràng buộc:** `1 <= arr.length <= 4·10^4`; `0 <= arr[i] <= 10^9`.

In [1]:
import itertools
import random
import time
from itertools import groupby
from typing import List

examples = [([9, 4, 2, 10, 7, 8, 8, 1, 9], 5), ([4, 8, 12, 16], 2), ([100], 1)]
for a, want in examples:
    print(f"arr = {a} -> {want}")

arr = [9, 4, 2, 10, 7, 8, 8, 1, 9] -> 5
arr = [4, 8, 12, 16] -> 2
arr = [100] -> 1


## 3. Kiến thức nền

### 3.1 Định nghĩa turbulent theo đúng đề

Hàm `is_turbulent(arr, i, j)` kiểm tra đúng hai điều kiện của đề (theo chỉ số **tuyệt đối** `k`, chẵn/lẻ). Đây là oracle gốc cho cả notebook. Đoạn một phần tử (`i == j`) thoả điều kiện một cách tầm thường (không có `k` nào).

In [2]:
def is_turbulent(arr, i, j):
    """Đúng nguyên văn đề: hoặc (>, k lẻ; <, k chẵn) hoặc (>, k chẵn; <, k lẻ) với mọi i <= k < j."""
    cond1 = all((arr[k] > arr[k + 1]) if k % 2 == 1 else (arr[k] < arr[k + 1]) for k in range(i, j))
    cond2 = all((arr[k] > arr[k + 1]) if k % 2 == 0 else (arr[k] < arr[k + 1]) for k in range(i, j))
    return cond1 or cond2


a = [9, 4, 2, 10, 7, 8, 8, 1, 9]
print(is_turbulent(a, 1, 5), is_turbulent(a, 0, 5), is_turbulent(a, 4, 6), is_turbulent(a, 6, 6))

True False False True


### 3.2 Mảng dấu và chuỗi xen kẽ

`d[i] = sign(arr[i+1] − arr[i])`. Đoạn `arr[l..r]` (`l < r`) turbulent ⇔ `d[l], …, d[r−1]` đều khác `0` và `d[k] · d[k+1] = −1` với mọi `l ≤ k < r − 1`. Ta viết `sign(x)` bằng `(x > 0) - (x < 0)`.

In [3]:
def sign(x):
    return (x > 0) - (x < 0)


a = [9, 4, 2, 10, 7, 8, 8, 1, 9]
d = [sign(a[i + 1] - a[i]) for i in range(len(a) - 1)]
print("d =", d)
print("khớp định nghĩa: arr[1..5] <->", d[1:5], "xen kẽ khác 0:", all(d[k] * d[k + 1] == -1 for k in range(1, 4)))

d = [-1, -1, 1, -1, 1, 0, -1, 1]
khớp định nghĩa: arr[1..5] <-> [-1, 1, -1, 1] xen kẽ khác 0: True


### 3.3 Điều kiện biên

- `n = 1`: đáp án `1`.
- Mảng toàn số bằng nhau: mọi `d = 0`, đáp án `1`.
- Mảng đơn điệu nghiêm ngặt: mọi `d` cùng dấu, đáp án `2`.

In [4]:
for arr in ([5], [3, 3, 3], [1, 2, 3, 4], [4, 3, 2, 1], [1, 3, 2, 4, 3]):
    print(arr, "-> d =", [sign(arr[i + 1] - arr[i]) for i in range(len(arr) - 1)])

[5] -> d = []
[3, 3, 3] -> d = [0, 0]
[1, 2, 3, 4] -> d = [1, 1, 1]
[4, 3, 2, 1] -> d = [-1, -1, -1]
[1, 3, 2, 4, 3] -> d = [1, -1, 1, -1]


## 4. Ý tưởng nền tảng / trực giác

`arr = [9,4,2,10,7,8,8,1,9]` có `d = [-1,-1,+1,-1,+1,0,-1,+1]`. Tách theo chỗ **phá** quy tắc xen kẽ:

- `d[0] = -1`, `d[1] = -1`: cùng dấu ⇒ run mới bắt đầu từ `d[1]`;
- `d[1..4] = (-1,+1,-1,+1)`: xen kẽ, dài `4` ⇒ đoạn `arr[1..5]` dài `5`;
- `d[5] = 0`: phá run;
- `d[6..7] = (-1,+1)`: dài `2` ⇒ đoạn dài `3`.

Đáp án `max(1, 2, 5, 3) = 5` — mỗi run xen kẽ gồm `m` dấu cho đoạn dài `m + 1`.

In [5]:
d = [sign(a[i + 1] - a[i]) for i in range(len(a) - 1)]
runs, cur = [], 0
for i, v in enumerate(d):
    if v == 0:
        runs.append(cur); cur = 0
    elif cur and d[i - 1] * v == -1:
        cur += 1
    else:
        runs.append(cur); cur = 1
runs.append(cur)
print("độ dài các run xen kẽ:", [r for r in runs if r], "-> đáp án = max + 1 =", max(runs) + 1)

độ dài các run xen kẽ: [1, 4, 2] -> đáp án = max + 1 = 5


## 5. Brute force — thử mọi đoạn con

**Ý tưởng:** với mỗi điểm bắt đầu `i`, mở rộng `j` chừng nào đoạn `arr[i..j]` còn turbulent (kiểm tra bước cuối so với bước trước) và ghi nhận độ dài. `O(n²)`. Bản oracle `O(n³)` ở mục 12 thử **mọi** `(i, j)` bằng `is_turbulent` đúng định nghĩa.

In [6]:
def turbulent_brute(arr):
    n, best = len(arr), 1
    for i in range(n):
        for j in range(i + 1, n):
            if arr[j] == arr[j - 1]:
                break
            if j - 1 > i and (arr[j] - arr[j - 1] > 0) == (arr[j - 1] - arr[j - 2] > 0):
                break
            best = max(best, j - i + 1)
    return best


def turbulent_cubic(arr):
    """Oracle: thử MỌI đoạn (i, j) bằng định nghĩa."""
    n = len(arr)
    return max(j - i + 1 for i in range(n) for j in range(i, n) if is_turbulent(arr, i, j))


for a, want in examples:
    assert turbulent_brute(a) == turbulent_cubic(a) == want
print([turbulent_brute(a) for a, _ in examples])

[5, 2, 1]


In [7]:
# Code nộp LeetCode cho Brute force (O(n^2): quá chậm với n = 4·10^4)
class SolutionBrute:
    def maxTurbulenceSize(self, arr: List[int]) -> int:
        n, best = len(arr), 1
        for i in range(n):
            for j in range(i + 1, n):
                if arr[j] == arr[j - 1]:
                    break
                if j - 1 > i and (arr[j] > arr[j - 1]) == (arr[j - 1] > arr[j - 2]):
                    break
                best = max(best, j - i + 1)
        return best

**Độ phức tạp:** `O(n²)` thời gian, `O(1)` bộ nhớ. Với `n = 4·10^4` là `~8·10^8` bước ⇒ quá chậm.

**Nút thắt:** mỗi lần bắt đầu mới ta kiểm tra lại các cặp đã kiểm tra. Một đoạn turbulent dài nhất kết thúc tại `j` được suy ra từ đoạn dài nhất kết thúc tại `j − 1`.

## 6. Bước đột phá

Gọi `up[j]` là độ dài đoạn turbulent dài nhất kết thúc tại `j` với bước cuối **tăng** (`arr[j−1] < arr[j]`), `down[j]` với bước cuối **giảm**. Nếu `arr[j] > arr[j−1]` thì `up[j] = down[j−1] + 1` (phải ghép sau một bước giảm), `down[j] = 1`; ngược lại tương tự; nếu bằng nhau cả hai về `1`. Chỉ cần giữ hai số ⇒ `O(n)` thời gian, `O(1)` bộ nhớ.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** DP hai trạng thái (đáp án chính), **Cách 3** cửa sổ trượt, **Cách 4** biến đổi dấu theo chẵn lẻ + `groupby`, **Cách 5** chia để trị.

### Cách 2 — DP hai trạng thái `up` / `down`

**Giải thích từng đoạn:**

1. `up = down = 1` (đoạn một phần tử).
2. Duyệt `j ≥ 1`: nếu `arr[j] > arr[j−1]` thì `up, down = down + 1, 1`; nếu `arr[j] < arr[j−1]` thì `up, down = 1, up + 1`; nếu bằng nhau thì `up = down = 1`.
3. Cập nhật `best = max(best, up, down)`.

Chú ý phép gán đồng thời: `down + 1` dùng `down` **cũ**.

In [8]:
def turbulent_dp(arr):
    up = down = best = 1
    for j in range(1, len(arr)):
        if arr[j] > arr[j - 1]:
            up, down = down + 1, 1                # bước tăng phải nối sau bước giảm
        elif arr[j] < arr[j - 1]:
            up, down = 1, up + 1                  # bước giảm phải nối sau bước tăng
        else:
            up = down = 1                         # bằng nhau: phá đoạn
        best = max(best, up, down)
    return best


for a, want in examples:
    assert turbulent_dp(a) == want
print([turbulent_dp(a) for a, _ in examples])

[5, 2, 1]


In [9]:
# Code nộp LeetCode cho Cách 2
class SolutionDP:
    def maxTurbulenceSize(self, arr: List[int]) -> int:
        up = down = best = 1
        for j in range(1, len(arr)):
            if arr[j] > arr[j - 1]:
                up, down = down + 1, 1
            elif arr[j] < arr[j - 1]:
                up, down = 1, up + 1
            else:
                up = down = 1
            best = max(best, up, down)
        return best

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### Cách 3 — Cửa sổ trượt một lượt

**Ý tưởng:** giữ đầu cửa sổ `start`. Với mỗi `j ≥ 1`, `c = sign(arr[j] − arr[j−1])`:

- `c == 0`: đoạn hiện tại bị phá ⇒ `start = j`;
- nếu `j ≥ 2` và dấu `c` **bằng** dấu bước trước (`sign(arr[j−1] − arr[j−2])`) thì không xen kẽ ⇒ cửa sổ mới bắt đầu từ `j − 1`, tức `start = j − 1`;
- sau đó `best = max(best, j − start + 1)`.

In [10]:
def turbulent_window(arr):
    best, start = 1, 0
    for j in range(1, len(arr)):
        c = sign(arr[j] - arr[j - 1])
        if c == 0:
            start = j                                      # cặp bằng nhau: cửa sổ mới bắt đầu tại j
        elif j >= 2 and c == sign(arr[j - 1] - arr[j - 2]):
            start = j - 1                                  # hai bước cùng dấu: giữ lại cặp (j-1, j)
        best = max(best, j - start + 1)
    return best


for a, want in examples:
    assert turbulent_window(a) == want
print([turbulent_window(a) for a, _ in examples])

[5, 2, 1]


In [11]:
# Code nộp LeetCode cho Cách 3
class SolutionWindow:
    def maxTurbulenceSize(self, arr: List[int]) -> int:
        def sign(x):
            return (x > 0) - (x < 0)

        best, start = 1, 0
        for j in range(1, len(arr)):
            c = sign(arr[j] - arr[j - 1])
            if c == 0:
                start = j
            elif j >= 2 and c == sign(arr[j - 1] - arr[j - 2]):
                start = j - 1
            best = max(best, j - start + 1)
        return best

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### Cách 4 — Biến đổi dấu theo chẵn lẻ + `groupby`

**Ý tưởng:** xen kẽ dấu ⇔ `d[i] · (−1)^i` **không đổi** (và khác `0`) trên đoạn. Đặt `t[i] = d[i] · (−1)^i`; khi đó mọi đoạn turbulent tương ứng một run các giá trị **bằng nhau và khác 0** của `t`. Dùng `groupby` tìm run dài nhất (bỏ qua run toàn `0`), đáp án = độ dài + 1 (hoặc `1` nếu không có run nào).

In [12]:
def turbulent_groupby(arr):
    t = [sign(arr[i + 1] - arr[i]) * (1 if i % 2 == 0 else -1) for i in range(len(arr) - 1)]
    best_run = 0
    for value, grp in groupby(t):
        if value != 0:
            best_run = max(best_run, sum(1 for _ in grp))
    return best_run + 1


for a, want in examples:
    assert turbulent_groupby(a) == want
print([turbulent_groupby(a) for a, _ in examples])
arr1 = [9, 4, 2, 10, 7, 8, 8, 1, 9]
print("t của ví dụ 1:", [sign(arr1[i + 1] - arr1[i]) * (1 if i % 2 == 0 else -1) for i in range(8)])

[5, 2, 1]
t của ví dụ 1: [-1, 1, 1, 1, 1, 0, -1, -1]


In [13]:
# Code nộp LeetCode cho Cách 4
from itertools import groupby


class SolutionGroupBy:
    def maxTurbulenceSize(self, arr: List[int]) -> int:
        def sign(x):
            return (x > 0) - (x < 0)

        t = [sign(arr[i + 1] - arr[i]) * (1 if i % 2 == 0 else -1) for i in range(len(arr) - 1)]
        best = 0
        for v, grp in groupby(t):
            if v != 0:
                best = max(best, len(list(grp)))
        return best + 1

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (mảng `t`).

### Cách 5 — Chia để trị (gộp tiền tố/hậu tố)

**Ý tưởng:** trên mảng dấu `d` (độ dài `m = n − 1`), mỗi đoạn `d[l..r]` lưu bộ `(best, pre, suf, len)`: run xen kẽ khác 0 dài nhất, độ dài tiền tố xen kẽ, hậu tố xen kẽ, và độ dài đoạn. Ghép hai nửa `A`, `B` liền kề: nếu **mối nối hợp lệ** (`d` cuối của `A` và `d` đầu của `B` khác 0 và trái dấu) thì có thể có run qua điểm giữa: `suf_A + pre_B`; `pre` mới = `pre_A` (+ `pre_B` nếu `pre_A` phủ toàn `A`); `suf` mới tương tự. Đáp án = `best + 1`.

In [14]:
def turbulent_dc(arr):
    n = len(arr)
    if n == 1:
        return 1
    d = [sign(arr[i + 1] - arr[i]) for i in range(n - 1)]

    def solve(l, r):                                   # trả (best, pre, suf, length) cho d[l..r]
        if l == r:
            ok = 1 if d[l] != 0 else 0
            return (ok, ok, ok, 1)
        m = (l + r) // 2
        a, b = solve(l, m), solve(m + 1, r)
        join = d[m] != 0 and d[m + 1] != 0 and d[m] * d[m + 1] == -1      # mối nối xen kẽ?
        best = max(a[0], b[0])
        pre, suf = a[1], b[2]
        if join:
            best = max(best, a[2] + b[1])
            if a[1] == a[3]:
                pre = a[3] + b[1]
            if b[2] == b[3]:
                suf = b[3] + a[2]
        return (best, pre, suf, a[3] + b[3])

    return solve(0, n - 2)[0] + 1


for a, want in examples:
    assert turbulent_dc(a) == want
print([turbulent_dc(a) for a, _ in examples])

[5, 2, 1]


In [15]:
# Code nộp LeetCode cho Cách 5
class SolutionDC:
    def maxTurbulenceSize(self, arr: List[int]) -> int:
        n = len(arr)
        if n == 1:
            return 1
        d = [(arr[i + 1] > arr[i]) - (arr[i + 1] < arr[i]) for i in range(n - 1)]

        def solve(l, r):
            if l == r:
                ok = 1 if d[l] else 0
                return (ok, ok, ok, 1)
            m = (l + r) // 2
            a, b = solve(l, m), solve(m + 1, r)
            join = d[m] * d[m + 1] == -1
            best, pre, suf = max(a[0], b[0]), a[1], b[2]
            if join:
                best = max(best, a[2] + b[1])
                if a[1] == a[3]:
                    pre = a[3] + b[1]
                if b[2] == b[3]:
                    suf = b[3] + a[2]
            return (best, pre, suf, a[3] + b[3])

        return solve(0, n - 2)[0] + 1

**Độ phức tạp:** `T(m) = 2T(m/2) + O(1)` ⇒ `O(n)` thời gian; `O(n)` bộ nhớ (mảng `d`) + `O(log n)` ngăn xếp.

## 8. TRACE từng bước

**Trace Cách 2 (DP)** với `arr = [9,4,2,10,7,8,8,1,9]`:

| `j` | `arr[j-1] → arr[j]` | bước | `up` | `down` | `best` |
|---|---|---|---|---|---|
| 0 | — | — | 1 | 1 | 1 |
| 1 | `9 → 4` | giảm | 1 | 2 | 2 |
| 2 | `4 → 2` | giảm | 1 | 2 | 2 |
| 3 | `2 → 10` | tăng | 3 | 1 | 3 |
| 4 | `10 → 7` | giảm | 1 | 4 | 4 |
| 5 | `7 → 8` | tăng | 5 | 1 | 5 |
| 6 | `8 → 8` | bằng | 1 | 1 | 5 |
| 7 | `8 → 1` | giảm | 1 | 2 | 5 |
| 8 | `1 → 9` | tăng | 3 | 1 | 5 |

Đáp án `5` ✓. Hàm trace in từng bước:

In [16]:
def trace_dp(arr):
    print("arr =", arr)
    up = down = best = 1
    print(f"  j=0: up={up} down={down} best={best}")
    for j in range(1, len(arr)):
        if arr[j] > arr[j - 1]:
            up, down, step = down + 1, 1, "tăng"
        elif arr[j] < arr[j - 1]:
            up, down, step = 1, up + 1, "giảm"
        else:
            up = down = 1
            step = "bằng"
        best = max(best, up, down)
        print(f"  j={j}: {arr[j - 1]} -> {arr[j]} ({step}): up={up} down={down} best={best}")
    return best


assert trace_dp([9, 4, 2, 10, 7, 8, 8, 1, 9]) == 5

arr = [9, 4, 2, 10, 7, 8, 8, 1, 9]
  j=0: up=1 down=1 best=1
  j=1: 9 -> 4 (giảm): up=1 down=2 best=2
  j=2: 4 -> 2 (giảm): up=1 down=2 best=2
  j=3: 2 -> 10 (tăng): up=3 down=1 best=3
  j=4: 10 -> 7 (giảm): up=1 down=4 best=4
  j=5: 7 -> 8 (tăng): up=5 down=1 best=5
  j=6: 8 -> 8 (bằng): up=1 down=1 best=5
  j=7: 8 -> 1 (giảm): up=1 down=2 best=5
  j=8: 1 -> 9 (tăng): up=3 down=1 best=5


**Trace Cách 3 (cửa sổ trượt)** trên cùng mảng: `start` chuyển `0 → 1` tại `j = 2` (hai bước giảm liên tiếp ⇒ giữ cặp `(1, 2)`), `start → 6` tại `j = 6` (bằng nhau), `start → 6` giữ ở `j = 7, 8`. Độ dài lớn nhất `j − start + 1 = 5 − 1 + 1 = 5` tại `j = 5`.

In [17]:
def trace_window(arr):
    best, start = 1, 0
    print("arr =", arr)
    for j in range(1, len(arr)):
        c = sign(arr[j] - arr[j - 1])
        note = ""
        if c == 0:
            start, note = j, "bằng nhau => start = j"
        elif j >= 2 and c == sign(arr[j - 1] - arr[j - 2]):
            start, note = j - 1, "cùng dấu bước trước => start = j-1"
        best = max(best, j - start + 1)
        print(f"  j={j}: start={start} độ dài={j - start + 1} best={best} {note}")
    return best


assert trace_window([9, 4, 2, 10, 7, 8, 8, 1, 9]) == 5

arr = [9, 4, 2, 10, 7, 8, 8, 1, 9]
  j=1: start=0 độ dài=2 best=2 
  j=2: start=1 độ dài=2 best=2 cùng dấu bước trước => start = j-1
  j=3: start=1 độ dài=3 best=3 
  j=4: start=1 độ dài=4 best=4 
  j=5: start=1 độ dài=5 best=5 
  j=6: start=6 độ dài=1 best=5 bằng nhau => start = j
  j=7: start=6 độ dài=2 best=5 
  j=8: start=6 độ dài=3 best=5 


## 9. Code nộp LeetCode

Chọn **Cách 2** (DP hai trạng thái): ngắn, `O(1)` bộ nhớ, dễ chứng minh. Giữ nguyên `class Solution`, `maxTurbulenceSize(self, arr)`:

In [18]:
class Solution:
    def maxTurbulenceSize(self, arr: List[int]) -> int:
        up = down = best = 1                      # up/down: độ dài tốt nhất kết thúc tại j với bước cuối tăng/giảm
        for j in range(1, len(arr)):
            if arr[j] > arr[j - 1]:
                up, down = down + 1, 1            # nối sau một bước giảm (down cũ)
            elif arr[j] < arr[j - 1]:
                up, down = 1, up + 1              # nối sau một bước tăng (up cũ)
            else:
                up = down = 1                     # bằng nhau: phá đoạn
            best = max(best, up, down)
        return best


print([Solution().maxTurbulenceSize(a) for a, _ in examples])
assert all(Solution().maxTurbulenceSize(a) == w for a, w in examples)

[5, 2, 1]


## 10. Vì sao đúng

**(a) Đặc trưng bằng mảng dấu.** Với `l < r`: `arr[l..r]` turbulent ⇔ `d[l..r−1]` đều khác `0` và `d[k]·d[k+1] = −1`. Thật vậy, điều kiện thứ nhất của đề nói `d[k] = +1` khi `k` chẵn, `−1` khi `k` lẻ (với mọi `k ∈ [l, r−1]`); điều kiện thứ hai là đảo dấu. Cả hai đều có nghĩa các dấu liên tiếp là **khác 0 và đổi dấu từng bước**; ngược lại, dãy dấu khác 0 đổi dấu từng bước có dạng `(s, −s, s, …)`, mà `s = d[l]` thuộc một trong hai mẫu (theo chẵn lẻ của `l`). Đoạn `l = r` luôn turbulent.

**(b) DP đúng.** Gọi `U[j]`, `D[j]` là độ dài dài nhất của đoạn turbulent kết thúc tại `j` mà bước cuối là tăng / giảm (hoặc `1` nếu không tồn tại bước như vậy). Nếu `arr[j] > arr[j−1]`: đoạn kết thúc tại `j` với bước cuối tăng có dạng `[đoạn kết thúc tại j−1 với bước cuối giảm] + arr[j]` (hoặc chỉ `{arr[j−1], arr[j]}`, độ dài 2, cũng nằm trong công thức vì `D[j−1] ≥ 1`), nên `U[j] = D[j−1] + 1`; còn `D[j] = 1` vì bước cuối không thể là giảm. Tương tự cho `arr[j] < arr[j−1]`. Nếu bằng nhau thì cặp `(j−1, j)` không thể nằm trong đoạn nên cả hai là `1`. Đáp án = `max_j max(U[j], D[j])`. Phép gán đồng thời dùng giá trị **cũ**.

**(c) Cửa sổ trượt đúng.** Bất biến: `[start, j]` là đoạn turbulent dài nhất kết thúc tại `j`. Nếu `c = 0`, đoạn tối đa kết thúc tại `j` chỉ gồm `{j}`: `start = j`. Nếu `c` cùng dấu với bước trước thì cặp `(j−1, j)` và `(j−2, j−1)` không xen kẽ, đoạn dài nhất kết thúc tại `j` chỉ chứa `(j−1, j)`: `start = j − 1`. Ngược lại giữ `start` (đoạn kết thúc tại `j−1` kéo dài thêm).

**(d) Cách 4.** `d[k]·d[k+1] = −1 ⇔ d[k]·(−1)^k = d[k+1]·(−1)^{k+1}` (khi cả hai khác 0): nhân thêm `(−1)^k` biến "đổi dấu liên tiếp" thành "bằng nhau". Do đó run các `t` bằng nhau và khác `0` chính là các run xen kẽ của `d`.

**(e) Cách 5.** Run xen kẽ khác 0 dài nhất nằm trọn trong nửa trái, nửa phải, hoặc cắt qua mối nối `(m, m+1)`. Run cắt qua gồm một hậu tố xen kẽ của nửa trái và một tiền tố xen kẽ của nửa phải, và mối nối phải xen kẽ; khi đó độ dài tối đa là `suf_A + pre_B`. Tiền tố/hậu tố mới cập nhật bằng cách nối khi nửa kia được phủ trọn.

**(f) Đáp án `1` cho `n = 1` / mọi `d = 0`; đơn điệu cho `2`** — được phủ bởi công thức `max(...) + 1` hoặc khởi tạo `best = 1`.

**(g) Kiểm chứng bằng máy.** Mọi cách khớp oracle `turbulent_cubic` (thử **mọi** đoạn bằng `is_turbulent` đúng định nghĩa) trên **mọi** mảng độ dài `≤ 7` với phần tử trong `{0,1,2}` và mọi mảng độ dài `≤ 6` trong `{0,1,2,3}`; khớp brute `O(n²)` trên mảng ngẫu nhiên tới `n = 3000`; các cách `O(n)` khớp nhau tới `n = 4·10^4` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi đoạn | `O(n²)` | `O(1)` | oracle `n ≲ 3000` |
| 2. DP `up`/`down` | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Cửa sổ trượt | `O(n)` | `O(1)` | cách "kinh điển" |
| 4. Dấu theo chẵn lẻ + `groupby` | `O(n)` | `O(n)` | dạng khai báo |
| 5. Chia để trị | `O(n)` | `O(n)` | minh hoạ gộp tiền tố/hậu tố |

Đo thực tế `n = 4·10^4`:

In [19]:
def bench(fn, arr, repeat=5):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(arr)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(978)
big = [rng.randint(0, 10 ** 9) for _ in range(40000)]
for name, fn in [("dp", turbulent_dp), ("window", turbulent_window), ("groupby", turbulent_groupby), ("dc", turbulent_dc)]:
    print(f"{name:8s}: {bench(fn, big) * 1e3:7.2f} ms (n = 4e4)")
print(f"brute   : {bench(turbulent_brute, big[:3000], repeat=1) * 1e3:7.1f} ms (n = 3000)")

dp      :    4.46 ms (n = 4e4)


window  :   10.36 ms (n = 4e4)


groupby :    9.42 ms (n = 4e4)


dc      :   21.98 ms (n = 4e4)
brute   :     2.1 ms (n = 3000)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ với oracle định nghĩa, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [20]:
IMPLS = {
    "dp": turbulent_dp,
    "window": turbulent_window,
    "groupby": turbulent_groupby,
    "dc": turbulent_dc,
    "brute": turbulent_brute,
    "Solution": lambda a: Solution().maxTurbulenceSize(list(a)),
    "SolutionDP": lambda a: SolutionDP().maxTurbulenceSize(list(a)),
    "SolutionWindow": lambda a: SolutionWindow().maxTurbulenceSize(list(a)),
    "SolutionGroupBy": lambda a: SolutionGroupBy().maxTurbulenceSize(list(a)),
    "SolutionDC": lambda a: SolutionDC().maxTurbulenceSize(list(a)),
    "SolutionBrute": lambda a: SolutionBrute().maxTurbulenceSize(list(a)),
}
SLOW = {"brute", "SolutionBrute"}
fast = [n for n in IMPLS if n not in SLOW]


def check(arr, want=None, names=None, slow_limit=3000):
    if want is None:
        want = turbulent_cubic(arr) if len(arr) <= 25 else turbulent_brute(arr)
    for name in (names or IMPLS):
        if name in SLOW and len(arr) > slow_limit:
            continue
        assert IMPLS[name](arr) == want, (name, arr[:12], IMPLS[name](arr), want)


for a, want in examples:
    assert turbulent_cubic(a) == want
    check(a, want)
print("ví dụ đề: oracle định nghĩa và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle định nghĩa và 11 cài đặt đúng


### B. Edge case

- `n = 1` (kể cả giá trị `0` và `10^9`).
- `n = 2`: bằng nhau (`1`) / khác nhau (`2`).
- Toàn số bằng nhau.
- Đơn điệu tăng / giảm nghiêm ngặt (`2`).
- Zigzag hoàn hảo (toàn mảng turbulent).
- Hai bước cùng dấu rồi xen kẽ.
- Cặp bằng nhau chia đôi.
- Đoạn turbulent ở đầu, ở cuối, ở giữa.
- Giá trị cực đại `10^9` và `0` xen kẽ.

In [21]:
edge = [
    [0], [10 ** 9], [1, 1], [1, 2], [2, 1], [3, 3, 3, 3], [1, 2, 3, 4, 5], [5, 4, 3, 2, 1], [1, 3, 2, 4, 3, 5, 4],
    [3, 1, 2, 4, 3, 5], [1, 1, 2, 1, 2], [2, 1, 2, 1, 1, 2, 1, 2, 1], [5, 1, 5, 1, 5, 1], [0, 10 ** 9] * 8, [10 ** 9, 0] * 8,
    [1, 2, 2, 3], [4, 8, 12, 16], [9, 4, 2, 10, 7, 8, 8, 1, 9], [1, 2, 1, 1, 2, 1], [1, 2, 3, 2, 3, 2, 1, 2],
]
for arr in edge:
    check(arr)
print(len(edge), "edge case: mọi cài đặt đúng")
print("zigzag 12 phần tử:", Solution().maxTurbulenceSize([1, 3] * 6), "| tăng nghiêm ngặt:", Solution().maxTurbulenceSize(list(range(10))))

20 edge case: mọi cài đặt đúng
zigzag 12 phần tử: 12 | tăng nghiêm ngặt: 2


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [22]:
def bug_equal_ok(arr):                                 # coi cặp bằng nhau là hợp lệ (<=, >=)
    up = down = best = 1
    for j in range(1, len(arr)):
        if arr[j] >= arr[j - 1]:
            up, down = down + 1, 1
        else:
            up, down = 1, up + 1
        best = max(best, up, down)
    return best


def bug_zero_for_flat(arr):                            # trả 0 khi không có cặp khác nhau (mảng 1 phần tử / toàn bằng nhau)
    d = [sign(arr[i + 1] - arr[i]) for i in range(len(arr) - 1)]
    best = cur = 0
    for i, v in enumerate(d):
        if v == 0:
            cur = 0
        elif cur and d[i - 1] * v == -1:
            cur += 1
        else:
            cur = 1
        best = max(best, cur)
    return best + 1 if best else 0


def bug_window_start_i(arr):                           # reset start = j (thay vì j - 1) khi hai bước cùng dấu
    best, start = 1, 0
    for j in range(1, len(arr)):
        c = sign(arr[j] - arr[j - 1])
        if c == 0 or (j >= 2 and c == sign(arr[j - 1] - arr[j - 2])):
            start = j
        best = max(best, j - start + 1)
    return best


def bug_count_pairs(arr):                              # trả số bước (cặp) thay vì số phần tử
    return turbulent_dp(arr) - 1


def bug_only_up_first(arr):                            # chỉ xét đoạn bắt đầu bằng bước tăng
    best, cur = 1, 1
    expect_up = True
    for j in range(1, len(arr)):
        if (arr[j] > arr[j - 1]) == expect_up and arr[j] != arr[j - 1]:
            cur += 1
            expect_up = not expect_up
        else:
            cur = 2 if arr[j] > arr[j - 1] else 1
            expect_up = False if arr[j] > arr[j - 1] else True
        best = max(best, cur)
    return best


def bug_dp_no_reset_on_equal(arr):                     # bằng nhau: giữ nguyên up/down (không reset)
    up = down = best = 1
    for j in range(1, len(arr)):
        if arr[j] > arr[j - 1]:
            up, down = down + 1, 1
        elif arr[j] < arr[j - 1]:
            up, down = 1, up + 1
        best = max(best, up, down)
    return best


def bug_compare_two_back(arr):                         # so arr[j] với arr[j-2] thay vì arr[j-1]
    best, cur = 1, 1
    for j in range(2, len(arr)):
        if (arr[j] - arr[j - 1]) * (arr[j - 1] - arr[j - 2]) < 0:
            cur += 1
        else:
            cur = 2
        best = max(best, cur)
    return best


BUGS = [bug_equal_ok, bug_zero_for_flat, bug_window_start_i, bug_count_pairs, bug_only_up_first, bug_dp_no_reset_on_equal, bug_compare_two_back]
pool = [list(a) for n in range(1, 7) for a in itertools.product(range(3), repeat=n)]
for bug in BUGS:
    found = next((a for a in pool if bug(a) != turbulent_cubic(a)), None)
    assert found is not None, bug.__name__
    print(f"{bug.__name__:26s} sai trên {found}: ra {bug(found)}, đúng {turbulent_cubic(found)}")

bug_equal_ok               sai trên [0, 0]: ra 2, đúng 1
bug_zero_for_flat          sai trên [0]: ra 0, đúng 1
bug_window_start_i         sai trên [0, 1, 2, 0]: ra 2, đúng 3
bug_count_pairs            sai trên [0]: ra 0, đúng 1
bug_only_up_first          sai trên [1, 0]: ra 1, đúng 2
bug_dp_no_reset_on_equal   sai trên [0, 1, 1, 0]: ra 3, đúng 2
bug_compare_two_back       sai trên [0, 1]: ra 1, đúng 2


### D. Vét cạn mọi mảng nhỏ với oracle định nghĩa

Oracle `turbulent_cubic`: thử **mọi** đoạn `(i, j)` bằng `is_turbulent` (đúng hai điều kiện của đề theo chỉ số tuyệt đối). Mọi mảng độ dài `1..7` với phần tử trong `{0,1,2}` (`3^1 + … + 3^7 = 3 279` mảng) và độ dài `1..6` với phần tử trong `{0,1,2,3}` (`4^1 + … + 4^6 = 5 460` mảng). Nhiều hòa, nhiều run, mọi vị trí chẵn/lẻ của điểm bắt đầu.

In [23]:
t0 = time.perf_counter()
cnt = 0
for values, max_n in ((3, 7), (4, 6)):
    for n in range(1, max_n + 1):
        for arr in itertools.product(range(values), repeat=n):
            arr = list(arr)
            want = turbulent_cubic(arr)
            for name in IMPLS:
                assert IMPLS[name](arr) == want, (name, arr, IMPLS[name](arr), want)
            cnt += 1
print(cnt, "mảng nhỏ (3^≤7 và 4^≤6):", len(IMPLS), "cài đặt khớp oracle định nghĩa;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 3279 + 5460

8739 mảng nhỏ (3^≤7 và 4^≤6): 11 cài đặt khớp oracle định nghĩa; 0.3 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Khớp brute `O(n²)` (với `n ≤ 3000`); (2) đảo mảng không đổi đáp án; (3) `v → C − v` (đảo thứ tự giá trị) không đổi đáp án; (4) `1 ≤ đáp án ≤ n`; (5) `đáp án ≥ 2 ⇔` có hai phần tử liền kề khác nhau; (6) cộng hằng / nhân hằng dương không đổi đáp án; (7) đáp án của mảng không bé hơn đáp án của mọi đoạn con.

In [24]:
rng = random.Random(20240978)
t0 = time.perf_counter()
for it in range(400):
    n = rng.randint(1, 3000)
    span = rng.choice([1, 2, 3, 10, 10 ** 9])
    arr = [rng.randint(0, span) for _ in range(n)]
    want = turbulent_brute(arr)                                                 # (1)
    for name in fast:
        assert IMPLS[name](arr) == want, (name, it)
    assert Solution().maxTurbulenceSize(arr[::-1]) == want                      # (2)
    assert Solution().maxTurbulenceSize([span - v for v in arr]) == want        # (3)
    assert 1 <= want <= n                                                       # (4)
    assert (want >= 2) == any(arr[i] != arr[i + 1] for i in range(n - 1))      # (5)
    assert Solution().maxTurbulenceSize([3 * v + 5 for v in arr]) == want       # (6)
    i = rng.randint(0, n - 1)
    j = rng.randint(i, n - 1)
    assert Solution().maxTurbulenceSize(arr[i:j + 1]) <= want                   # (7)
print("400 mảng ngẫu nhiên (n ≤ 3000, nhiều giá trị lặp): 7 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

400 mảng ngẫu nhiên (n ≤ 3000, nhiều giá trị lặp): 7 tính chất đúng; 2.1 s


### F. Giới hạn

`n = 4·10^4`: mảng zigzag hoàn hảo (đáp án `n`), đơn điệu (`2`), toàn bằng nhau (`1`), ngẫu nhiên, và một run dài ở giữa; các cách `O(n)` và `O(n log n)`/`O(n)` khớp nhau (brute `O(n²)` bỏ qua).

In [25]:
t0 = time.perf_counter()
n = 40000
rng = random.Random(97801)
zig = [0 if i % 2 == 0 else 10 ** 9 for i in range(n)]
mono = list(range(n))
flat = [7] * n
rand = [rng.randint(0, 10 ** 9) for _ in range(n)]
mid = [1] * 15000 + [(2 if i % 2 else 1) for i in range(20000)] + [3] * 5000
small_range = [rng.randint(0, 1) for _ in range(n)]
for arr, want in [(zig, n), (mono, 2), (flat, 1), (rand, None), (mid, None), (small_range, None)]:
    results = {name: IMPLS[name](arr) for name in fast}
    assert len(set(results.values())) == 1, results
    if want is not None:
        assert results["Solution"] == want
    print(f"  đáp án = {results['Solution']}")
print("n = 4·10^4: mọi cài đặt nhanh khớp nhau;", round(time.perf_counter() - t0, 2), "s")

  đáp án = 40000


  đáp án = 2
  đáp án = 1


  đáp án = 23


  đáp án = 20000
  đáp án = 15
n = 4·10^4: mọi cài đặt nhanh khớp nhau; 0.53 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: coi cặp bằng nhau là hợp lệ

In [26]:
arr = [2, 2, 2]
print("bug:", bug_equal_ok(arr), "| đúng:", Solution().maxTurbulenceSize(arr))
assert bug_equal_ok(arr) == 2 and Solution().maxTurbulenceSize(arr) == 1

bug: 2 | đúng: 1


**Vì sao sai:** đề yêu cầu **dấu so sánh đổi chiều** — hai phần tử bằng nhau không có dấu. **Đúng:** dùng `>` và `<` nghiêm ngặt, và reset khi bằng nhau.

### Lỗi 2: trả `0` cho mảng một phần tử / toàn bằng nhau

In [27]:
print("bug:", bug_zero_for_flat([100]), bug_zero_for_flat([3, 3, 3]), "| đúng:", Solution().maxTurbulenceSize([100]), Solution().maxTurbulenceSize([3, 3, 3]))
assert bug_zero_for_flat([100]) == 0 and Solution().maxTurbulenceSize([100]) == 1

bug: 0 0 | đúng: 1 1


**Vì sao sai:** một phần tử đơn lẻ luôn là đoạn turbulent (không có cặp nào vi phạm). **Đúng:** khởi tạo `best = 1`.

### Lỗi 3: reset cửa sổ sai chỗ

In [28]:
arr = [9, 4, 2, 10, 7, 8, 8, 1, 9]
print("bug:", bug_window_start_i(arr), "| đúng:", Solution().maxTurbulenceSize(arr))
assert bug_window_start_i(arr) != 5

bug: 4 | đúng: 5


**Vì sao sai:** khi hai bước liên tiếp cùng dấu, cặp `(j−1, j)` vẫn có thể bắt đầu một đoạn mới; đặt `start = j` làm mất phần tử `arr[j−1]`. **Đúng:** `start = j − 1` (nhưng `start = j` khi bằng nhau).

### Lỗi 4: nhầm số cặp với số phần tử (lệch 1)

In [29]:
print("bug:", bug_count_pairs(arr), "| đúng:", Solution().maxTurbulenceSize(arr))
assert bug_count_pairs(arr) == 4

bug: 4 | đúng: 5


**Vì sao sai:** run xen kẽ có `m` bước tương ứng `m + 1` phần tử. **Đúng:** đáp án = số bước + 1.

### Lỗi 5: chỉ xét một hướng bắt đầu

In [30]:
arr5 = [3, 2, 3, 2]
print("bug:", bug_only_up_first(arr5), "| đúng:", Solution().maxTurbulenceSize(arr5))
assert bug_only_up_first(arr5) == 3 and Solution().maxTurbulenceSize(arr5) == 4

bug: 3 | đúng: 4


**Vì sao sai:** đề có hai mẫu (bắt đầu bằng `<` hoặc `>`); đoạn dài nhất có thể bắt đầu bằng bước giảm. **Đúng:** hai trạng thái `up` / `down`.

### Lỗi 6 (bonus): không reset khi bằng nhau / so sánh với `arr[j−2]`

In [31]:
arr6 = [1, 2, 2, 1]
print("không reset:", bug_dp_no_reset_on_equal(arr6), "| so arr[j-2]:", bug_compare_two_back([1, 2, 2, 3, 2]), "| đúng:", Solution().maxTurbulenceSize(arr6), Solution().maxTurbulenceSize([1, 2, 2, 3, 2]))
assert bug_dp_no_reset_on_equal(arr6) == 3 and Solution().maxTurbulenceSize(arr6) == 2

không reset: 3 | so arr[j-2]: 3 | đúng: 2 3


**Vì sao sai:** cặp bằng nhau **phá** chuỗi; giữ `up/down` cũ nối hai đoạn qua chỗ bằng nhau. Và phải so **liền kề** (`arr[j−1]`), không phải `arr[j−2]`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đoạn con dài nhất mà từng cặp liền kề thoả quy tắc xen kẽ" → DP theo hai trạng thái (kết thúc bằng tăng/giảm) hoặc chuyển về mảng dấu + run dài nhất.

**Mẹo:**

1. Chuyển bài toán sang mảng dấu `d[i] = sign(arr[i+1] − arr[i])`.
2. "Xen kẽ dấu" ⇔ nhân `d[i]` với `(−1)^i` để thành "bằng nhau".
3. DP hai trạng thái: gán đồng thời dùng giá trị cũ.
4. Khởi tạo `best = 1` (đoạn một phần tử).
5. Cặp bằng nhau luôn reset.
6. Oracle: thử mọi `(i, j)` bằng định nghĩa nguyên văn của đề.

**Họ bài cùng pattern:** Wiggle Subsequence (376), Longest Mountain in Array (845), Maximum Subarray (53), Best Time to Buy and Sell Stock II (122), Maximum Alternating Subsequence Sum (1911), Longest Continuous Increasing Subsequence (674).

## 15. Biến thể

1. **Trả về đoạn** (vị trí bắt đầu và độ dài), không chỉ độ dài.
2. **Wiggle Subsequence (376):** dãy con (không nhất thiết liền kề) xen kẽ dài nhất.
3. **Đếm mọi đoạn turbulent** (kể cả độ dài 1): bằng `Σ_j max(up_j, down_j)`.

Cài đặt và đối chiếu vét cạn:

In [32]:
def turbulent_segment(arr):
    """Biến thể 1: (start, length) của một đoạn turbulent dài nhất."""
    best, start, best_start = 1, 0, 0
    for j in range(1, len(arr)):
        c = sign(arr[j] - arr[j - 1])
        if c == 0:
            start = j
        elif j >= 2 and c == sign(arr[j - 1] - arr[j - 2]):
            start = j - 1
        if j - start + 1 > best:
            best, best_start = j - start + 1, start
    return best_start, best


def wiggle_length(nums):
    """Biến thể 2 (LeetCode 376): đếm số lần đổi chiều của dãy khác nhau liền kề + 1."""
    up = down = 1
    for j in range(1, len(nums)):
        if nums[j] > nums[j - 1]:
            up = down + 1
        elif nums[j] < nums[j - 1]:
            down = up + 1
    return max(up, down)


def wiggle_brute(nums):
    n, best = len(nums), 1 if nums else 0
    for mask in range(1, 1 << n):
        sub = [nums[i] for i in range(n) if mask >> i & 1]
        ds = [sign(sub[i + 1] - sub[i]) for i in range(len(sub) - 1)]
        if all(v != 0 for v in ds) and all(ds[i] * ds[i + 1] == -1 for i in range(len(ds) - 1)):
            best = max(best, len(sub))
    return best


def count_turbulent(arr):
    """Biến thể 3: số đoạn turbulent (kể cả độ dài 1) = tổng độ dài tốt nhất kết thúc tại mỗi vị trí."""
    up = down = 1
    total = 1
    for j in range(1, len(arr)):
        if arr[j] > arr[j - 1]:
            up, down = down + 1, 1
        elif arr[j] < arr[j - 1]:
            up, down = 1, up + 1
        else:
            up = down = 1
        total += max(up, down)
    return total


rng = random.Random(15)
for _ in range(400):
    n = rng.randint(1, 12)
    arr = [rng.randint(0, 3) for _ in range(n)]
    s, L = turbulent_segment(arr)
    assert L == turbulent_cubic(arr) and is_turbulent(arr, s, s + L - 1)                    # (1)
    assert wiggle_length(arr) == wiggle_brute(arr)                                          # (2)
    assert count_turbulent(arr) == sum(1 for i in range(n) for j in range(i, n) if is_turbulent(arr, i, j))   # (3)
print("đoạn cụ thể, wiggle subsequence, đếm mọi đoạn: khớp vét cạn")

đoạn cụ thể, wiggle subsequence, đếm mọi đoạn: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi đoạn | mở rộng từ từng `i` | `O(n²)` | `O(1)` | oracle `n ≲ 3000` |
| 2. DP `up`/`down` | kết thúc bằng tăng/giảm | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Cửa sổ trượt | reset `start` đúng chỗ | `O(n)` | `O(1)` | cách kinh điển |
| 4. Dấu × (−1)^i + `groupby` | xen kẽ → bằng nhau | `O(n)` | `O(n)` | dạng khai báo |
| 5. Chia để trị | gộp tiền tố/hậu tố | `O(n)` | `O(n)` | khung tổng quát cho "run dài nhất" |

Mọi cách khớp oracle định nghĩa trên toàn bộ mảng nhỏ (mục 12D) và khớp nhau tới `n = 4·10^4`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [33]:
print(Solution().maxTurbulenceSize([1, 2, 1, 2, 1]), Solution().maxTurbulenceSize([2, 2, 2]), Solution().maxTurbulenceSize([1, 3, 2, 2, 4, 3]),
      Solution().maxTurbulenceSize([5, 4, 3, 4, 3, 4]))

5 1 3 5


### Bài 2 (tính tay)

`arr = [3, 1, 4, 1, 5, 9, 2, 6]`. (a) Tính mảng dấu `d`. (b) Tìm các run xen kẽ. (c) Đáp án? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DP hai trạng thái:

In [34]:
def dp_todo(arr):
    up = down = best = 1
    for j in range(1, len(arr)):
        if arr[j] > arr[j - 1]:
            pass        # TODO 1: up, down = down + 1, 1
        elif arr[j] < arr[j - 1]:
            pass        # TODO 2: up, down = 1, up + 1
        else:
            pass        # TODO 3: up = down = 1
        best = max(best, up, down)
    return best


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(dp_todo(a) == w for a, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **cửa sổ trượt** (`start` và so sánh dấu liên tiếp):

In [35]:
class MySolution:
    def maxTurbulenceSize(self, arr):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().maxTurbulenceSize(list(a)) == w for a, w in examples)

### Bài 5 (mini challenge): Wiggle Subsequence (376)

Viết `wiggle_max_length(nums)` trả về độ dài dãy con (**không nhất thiết liền kề**) xen kẽ dài nhất: hiệu giữa các phần tử liên tiếp của dãy con xen kẽ dương/âm (khác 0). Kiểm bằng `wiggle_brute` ở mục 15.

In [36]:
def wiggle_max_length(nums):
    pass            # TODO


# Khi làm xong:
# assert wiggle_max_length([1, 7, 4, 9, 2, 5]) == 6 and wiggle_max_length([1, 17, 5, 10, 13, 15, 10, 5, 16, 8]) == 7

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: mỗi cặp bằng nhau phá đoạn; một phần tử đơn có độ dài `1`.
- Mức 2: `[1,2,1,2,1]` xen kẽ hoàn toàn.
- Mức 3: kết quả `5 1 3 5` (`[1,3,2,2,4,3]` bị cặp bằng nhau chia đôi, mỗi nửa dài `3`; `[5,4,3,4,3,4]` có đoạn `[4,3,4,3,4]` dài `5`).

**Bài 2**

- Mức 1: `d` có `7` phần tử (`n − 1`).
- Mức 2: `3→1` (−), `1→4` (+), `4→1` (−), `1→5` (+), `5→9` (+), `9→2` (−), `2→6` (+).
- Mức 3: `d = [-1,+1,-1,+1,+1,-1,+1]`; run xen kẽ `d[0..3]` dài `4` ⇒ đoạn `5`; run `d[4..6]` dài `3` ⇒ đoạn `4`; đáp án `5`.

**Bài 3**

- Mức 1: nối sau bước ngược chiều.
- Mức 2: `up, down = down + 1, 1`.
- Mức 3: giảm: `up, down = 1, up + 1`; bằng: `up = down = 1`.

**Bài 4**

- Mức 1: `best = 1`, `start = 0`.
- Mức 2: `c = sign(arr[j] - arr[j-1])`; `c == 0 ⇒ start = j`.
- Mức 3: `j >= 2 and c == sign(arr[j-1] - arr[j-2]) ⇒ start = j - 1`; `best = max(best, j - start + 1)`.

**Bài 5**

- Mức 1: chỉ cần đếm số lần đổi chiều (bỏ qua các bước cùng chiều).
- Mức 2: `up = down + 1` khi tăng, `down = up + 1` khi giảm.
- Mức 3: trả `max(up, down)` (khởi tạo `1`).

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [37]:
vals = (Solution().maxTurbulenceSize([1, 2, 1, 2, 1]), Solution().maxTurbulenceSize([2, 2, 2]), Solution().maxTurbulenceSize([1, 3, 2, 2, 4, 3]),
        Solution().maxTurbulenceSize([5, 4, 3, 4, 3, 4]))
print(vals)
assert vals == (5, 1, 3, 5)
for a, v in zip(([1, 2, 1, 2, 1], [2, 2, 2], [1, 3, 2, 2, 4, 3], [5, 4, 3, 4, 3, 4]), vals):
    assert v == turbulent_cubic(a)
print("Bài 1 OK")

(5, 1, 3, 5)
Bài 1 OK


### Bài 2

In [38]:
arr = [3, 1, 4, 1, 5, 9, 2, 6]
d = [sign(arr[i + 1] - arr[i]) for i in range(len(arr) - 1)]
print(d)
assert d == [-1, 1, -1, 1, 1, -1, 1]
assert Solution().maxTurbulenceSize(arr) == turbulent_cubic(arr) == 5
print("Bài 2 OK: đáp án 5 (đoạn [3,1,4,1,5])")

[-1, 1, -1, 1, 1, -1, 1]
Bài 2 OK: đáp án 5 (đoạn [3,1,4,1,5])


### Bài 3

In [39]:
def dp_answer(arr):
    up = down = best = 1
    for j in range(1, len(arr)):
        if arr[j] > arr[j - 1]:
            up, down = down + 1, 1
        elif arr[j] < arr[j - 1]:
            up, down = 1, up + 1
        else:
            up = down = 1
        best = max(best, up, down)
    return best


assert all(dp_answer(a) == w for a, w in examples)
rng = random.Random(3)
for _ in range(300):
    arr = [rng.randint(0, 3) for _ in range(rng.randint(1, 12))]
    assert dp_answer(arr) == turbulent_cubic(arr)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [40]:
class MySolutionAnswer:
    def maxTurbulenceSize(self, arr):
        def sgn(x):
            return (x > 0) - (x < 0)

        best, start = 1, 0
        for j in range(1, len(arr)):
            c = sgn(arr[j] - arr[j - 1])
            if c == 0:
                start = j
            elif j >= 2 and c == sgn(arr[j - 1] - arr[j - 2]):
                start = j - 1
            best = max(best, j - start + 1)
        return best


assert all(MySolutionAnswer().maxTurbulenceSize(list(a)) == w for a, w in examples)
rng = random.Random(4)
for _ in range(300):
    arr = [rng.randint(0, 3) for _ in range(rng.randint(1, 12))]
    assert MySolutionAnswer().maxTurbulenceSize(arr) == turbulent_cubic(arr)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [41]:
def wiggle_max_length_answer(nums):
    up = down = 1
    for j in range(1, len(nums)):
        if nums[j] > nums[j - 1]:
            up = down + 1
        elif nums[j] < nums[j - 1]:
            down = up + 1
    return max(up, down)


assert wiggle_max_length_answer([1, 7, 4, 9, 2, 5]) == 6 and wiggle_max_length_answer([1, 17, 5, 10, 13, 15, 10, 5, 16, 8]) == 7
rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(0, 4) for _ in range(rng.randint(1, 12))]
    assert wiggle_max_length_answer(nums) == wiggle_brute(nums)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đoạn con dài nhất với quy tắc xen kẽ giữa các bước liền kề → DP hai trạng thái):**

```python
class Solution:
    def maxTurbulenceSize(self, arr: List[int]) -> int:
        up = down = best = 1
        for j in range(1, len(arr)):
            if arr[j] > arr[j - 1]:
                up, down = down + 1, 1
            elif arr[j] < arr[j - 1]:
                up, down = 1, up + 1
            else:
                up = down = 1
            best = max(best, up, down)
        return best
```

**Checklist trước khi submit:**

- [ ] `best = 1` ban đầu (một phần tử vẫn là đoạn turbulent)?
- [ ] So sánh **nghiêm ngặt**; cặp bằng nhau reset `up = down = 1`?
- [ ] Gán đồng thời `up, down = down + 1, 1` (dùng giá trị cũ)?
- [ ] Cập nhật `best = max(best, up, down)` ở mỗi bước?
- [ ] Đáp án là số **phần tử** (không phải số cặp)?
- [ ] Đã thử `[9,4,2,10,7,8,8,1,9] → 5`, `[4,8,12,16] → 2`, `[100] → 1`, toàn bằng nhau?
- [ ] Đúng tên `class Solution`, `maxTurbulenceSize(self, arr)`, không in debug.